# Benchmarking hadal_flow

In [1]:
import hadal_flow
import tensorflow as tf
import timeit

context = hadal_flow.create_context64(
    log_n=10,
    main_moduli=[8556589057, 8388812801],
    plaintext_modulus=40961,
    scaling_factor=3,
    seed="test_seed",
)

secret_key = hadal_flow.create_key64(context)
rotation_key = hadal_flow.create_rotation_key64(context, secret_key)

a = tf.random.uniform([context.num_slots, 55555], dtype=tf.float32, maxval=10)
b = tf.random.uniform([55555, 333], dtype=tf.float32, maxval=10)
c = tf.random.uniform([2, context.num_slots], dtype=tf.float32, maxval=10)
d = tf.random.uniform([context.num_slots, 4444], dtype=tf.float32, maxval=10)

enc_a = hadal_flow.to_encrypted(a, secret_key, context)

2026-03-16 20:20:49.893125: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-16 20:20:52.249936: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


INFO: Generating key
INFO: Generating rotation key
INFO: Generating rotation key


In [2]:
def to_pt():
    return hadal_flow.to_shell_plaintext(a, context)

time = min(timeit.Timer(to_pt).repeat(repeat=3, number=1))
print(time)

0.1742473531048745


In [3]:
def enc():
    return hadal_flow.to_encrypted(d, secret_key, context)

time = min(timeit.Timer(enc).repeat(repeat=3, number=1))
print(time)

0.06979422480799258


In [4]:
def dec():
    return hadal_flow.to_tensorflow(enc_a, secret_key)

time = min(timeit.Timer(dec).repeat(repeat=3, number=1))
print(time)

0.236358264926821


In [5]:
def ct_ct_add():
    return enc_a + enc_a

time = min(timeit.Timer(ct_ct_add).repeat(repeat=3, number=1))
print(time)

0.1293677370995283


In [6]:
def ct_ct_sub():
    return enc_a - enc_a

time = min(timeit.Timer(ct_ct_sub).repeat(repeat=3, number=1))
print(time)

0.12651230487972498


In [7]:
def ct_ct_mul():
    return enc_a * 4

time = min(timeit.Timer(ct_ct_mul).repeat(repeat=3, number=1))
print(time)

0.1360173779539764


In [8]:
def ct_pt_add():
    return enc_a + a

time = min(timeit.Timer(ct_pt_add).repeat(repeat=3, number=1))
print(time)

0.3777763850521296


In [9]:
def ct_pt_mul():
    return enc_a * a

time = min(timeit.Timer(ct_pt_mul).repeat(repeat=3, number=1))
print(time)

0.4222509309183806


In [10]:
def ct_pt_matmul():
    return hadal_flow.matmul(enc_a, b)

time = min(timeit.Timer(ct_pt_matmul).repeat(repeat=3, number=1))
print(time)

11.817895537940785


In [11]:
def pt_ct_matmul():
    return hadal_flow.matmul(c, enc_a, rotation_key)

time = min(timeit.Timer(pt_ct_matmul).repeat(repeat=3, number=1))
print(time)

493.563799012918


In [12]:
def ct_roll():
    return hadal_flow.roll(enc_a, 2, rotation_key)

time = min(timeit.Timer(ct_roll).repeat(repeat=3, number=1))
print(time)

2.3143266271799803
